In [ ]:
# Silver 변환 노트북: Binance Bronze → Silver (4h 캔들 중심)
from pyspark.sql.functions import col, from_json, split, coalesce, to_date
from pyspark.sql.types import ArrayType, StringType
from delta.tables import DeltaTable

spark.sql("SET spark.sql.session.timeZone=UTC")  # 경계 정합

DAYS_BACK = 14
CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"
BRONZE  = f"{CATALOG}.{SCHEMA}.bronze_charts"
SILVER  = f"{CATALOG}.{SCHEMA}.silver_charts"

# ===== Spark / Delta Performance Configuration =====
# optimizeWrite: 커밋 전 소파일을 병합 → 소파일 누적 방지
spark.conf.set("spark.databricks.delta.optimizeWrite", "true")
# autoCompact: 쓰기 완료 후 백그라운드 컴팩션 자동 트리거
spark.conf.set("spark.databricks.delta.autoCompact", "true")
# AQE: 런타임 실행 통계를 기반으로 Spark가 쿼리 플랜을 동적으로 재조정
spark.conf.set("spark.sql.adaptive.enabled", "true")
# 셔플 후 소규모/빈 파티션을 동적으로 병합하여 Task 오버헤드 감소
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Sort-Merge Join에서 데이터 Skew를 자동 감지하고 서브태스크로 분할
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER} (
  symbol       STRING,
  interval     STRING,
  open_time    TIMESTAMP,
  open         DOUBLE,
  high         DOUBLE,
  low          DOUBLE,
  close        DOUBLE,
  volume       DOUBLE,
  unique_key   STRING,
  event_time   TIMESTAMP,
  dt           DATE
) USING DELTA
PARTITIONED BY (dt)
""")

bronze = spark.table(BRONZE)
if DAYS_BACK is not None:
    bronze = bronze.where(f"dt >= date_sub(current_date(), {DAYS_BACK})")

# 4시간봉만 사용한다면 초기에 필터
bronze = bronze.where("interval = '4h' OR unique_key LIKE '%|4h|%'")

arr = from_json(col("raw_json"), ArrayType(StringType()))
uk  = split(col("unique_key"), "\\|")

silver_df = (
    bronze
      # 브론즈에 컬럼이 있으면 우선 사용, 없으면 uk에서 보완
      .withColumn("symbol_eff",   coalesce(col("symbol"),   uk.getItem(0)))
      .withColumn("interval_eff", coalesce(col("interval"), uk.getItem(1)))
      .withColumn("open_time",  (arr.getItem(0).cast("long")/1000).cast("timestamp"))
      .withColumn("open",       arr.getItem(1).cast("double"))
      .withColumn("high",       arr.getItem(2).cast("double"))
      .withColumn("low",        arr.getItem(3).cast("double"))
      .withColumn("close",      arr.getItem(4).cast("double"))
      .withColumn("volume",     arr.getItem(5).cast("double"))
      .withColumn("dt",         to_date(col("open_time")))             # open_time 기준으로 통일
      .selectExpr(
          "symbol_eff as symbol",
          "interval_eff as interval",
          "open_time","open","high","low","close","volume",
          "unique_key","event_time","dt"
      )
      .dropDuplicates(["unique_key"])
      .repartition("dt")
)

target = DeltaTable.forName(spark, SILVER)
(target.alias("t")
  .merge(
    silver_df.alias("s"),
    "t.unique_key = s.unique_key AND t.dt = s.dt"
  )
  .whenMatchedUpdate(set={
      "symbol":     "s.symbol",
      "interval":   "s.interval",
      "open_time":  "s.open_time",
      "open":       "s.open",
      "high":       "s.high",
      "low":        "s.low",
      "close":      "s.close",
      "volume":     "s.volume",
      "event_time": "s.event_time",
      "dt":         "s.dt"
  })
  .whenNotMatchedInsertAll()
  .execute())

print(f"[SILVER] upsert complete: {SILVER}")

In [ ]:
# ===== DATA QUALITY CHECKS (silver_charts) =====
# MERGE 완료 후 최근 2일 데이터에 대해 데이터 계약(Data Contract) 검증 수행
# 임계값 초과 시 assert로 노트북을 즉시 중단하여 오염 데이터가 Gold로 전파되는 것을 방지
from pyspark.sql import functions as F

dq = spark.table(SILVER).where("dt >= date_sub(current_date(), 2)")

null_check = dq.select(
    F.count("*").alias("total_rows"),
    F.sum(F.col("close").isNull().cast("int")).alias("null_close"),
    F.sum(F.col("open_time").isNull().cast("int")).alias("null_open_time"),
    F.sum(F.col("volume").isNull().cast("int")).alias("null_volume"),
    F.sum((F.col("close") <= 0).cast("int")).alias("nonpositive_close"),
    F.sum((F.col("high") < F.col("low")).cast("int")).alias("high_lt_low"),
    F.min("open_time").alias("earliest"),
    F.max("open_time").alias("latest"),
).collect()[0]

print(f"[DQ] silver_charts (최근 2일)")
print(f"  total_rows       : {null_check['total_rows']}")
print(f"  null_close       : {null_check['null_close']}")
print(f"  null_open_time   : {null_check['null_open_time']}")
print(f"  null_volume      : {null_check['null_volume']}")
print(f"  nonpositive_close: {null_check['nonpositive_close']}")
print(f"  high < low       : {null_check['high_lt_low']}")
print(f"  range            : {null_check['earliest']} → {null_check['latest']}")

# 핵심 품질 계약 위반 시 파이프라인 즉시 중단
assert null_check["null_close"] == 0, \
    f"[DQ FAIL] close 컬럼에 null 발생: {null_check['null_close']}건"
assert null_check["nonpositive_close"] == 0, \
    f"[DQ FAIL] close <= 0 비정상 가격 발생: {null_check['nonpositive_close']}건"
assert null_check["high_lt_low"] == 0, \
    f"[DQ FAIL] high < low 불가능한 OHLC 관계 발생: {null_check['high_lt_low']}건"

print("[DQ] 모든 품질 체크 통과 ✓")